IMPORTING ENVIRONMENT VARIABLES

In [1]:
import sys
print(sys.executable)

c:\Users\parth\OneDrive\Desktop\DataHub\react_agent\.venv\Scripts\python.exe


In [2]:

from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

print("Groq key loaded:", groq_api_key is not None)
print("Gemini key loaded:", gemini_api_key is not None)

Groq key loaded: True
Gemini key loaded: True


LOADING THE PDF

In [4]:
pdf_path = "../data/ABC_Company_RAG_Learning_Document.pdf"
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("Pages:", len(documents))

<ipython-input-4-2984e5fd0451>:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 3


In [5]:
print(documents[0].page_content)

ABC Technologies Pvt. Ltd.
 Fictional Company Knowledge Base — RAG Learning Dataset
1. Company Overview
ABC Technologies Pvt. Ltd. is a fictional software company created specifically for learning and demonstrating
Retrieval-Augmented Generation (RAG). The company builds cloud-based business applications for small and
medium-sized organizations. Its main office is located in Chennai, India, and it has approximately 180 employees.
2. Products and Services
 Product / Service
Description
ABC Analytics
A business intelligence platform for dashboards, reports, KPIs, and scheduled data analysis.
ABC Cloud
A cloud hosting and storage platform used by customers to deploy internal applications.
ABC Support
A customer-support service providing technical assistance, incident tracking, and product guidance.
Data Engineering Services
ETL pipelines, data integration, data-quality checks, and warehouse development.
3. Company Departments
Engineering: Develops and maintains ABC's software products. Th

LOADING EMBEDDING MODEL

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("Embedding model created")

Embedding model created


CREATING CHUNKS OF THE DOCUMENT

In [12]:
from langchain_experimental.text_splitter import SemanticChunker
text_splitter=SemanticChunker(embeddings,breakpoint_threshold_type="percentile")
chunks = text_splitter.split_documents(documents)
print("Original pages:", len(documents))
print("Number of chunks:", len(chunks))

Original pages: 3
Number of chunks: 7


STORING CHUNKS AND ITS VECTORS IN VECTOR DATABASE

In [13]:
from langchain_chroma import Chroma
chroma_path="./chroma_db"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="abc_company",
    persist_directory=chroma_path
)

print("ChromaDB created successfully")

ChromaDB created successfully


CONNECTING TO GROQ LLM FOR ANSWERING THE QUERIES

In [21]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b",api_key=groq_api_key,temperature=0.2)

UDF FOR GETTING CONTEXT(FROM VECTOR_DB) AND ANSWERING USER QUERIES USING LLM

In [22]:
def ask_question(question): 
    results = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n\n".join(
        doc.page_content
        for doc in results
    )
    prompt = f"""
    Answer the user's question using the context below.
    Context:
    {context}
    User Question:
    {question}
    If the answer is not present in the context, say that you don't know based on the provided document.
    """
    response=llm.invoke(prompt)
    return response.content


In [23]:
question=input("Enter your question: ")
answer=ask_question(question)
print("Agent answer: ",answer)

Agent answer:  **Products and Services offered by ABC Technologies Pvt. Ltd.:**

| Product / Service | Description |
|-------------------|-------------|
| **ABC Analytics** | A business‑intelligence platform that enables dashboards, reports, KPIs, and scheduled data analysis. |
| **ABC Cloud** | A cloud hosting and storage platform used by customers to deploy internal applications, with object storage, backups, and monitoring. |
| **ABC Support** | A customer‑support service that provides technical assistance, incident tracking, and product guidance. |
| **Data Engineering Services** | End‑to‑end data solutions including ETL pipelines, data integration, data‑quality checks, and data‑warehouse development. |

These are the primary products and services highlighted in the company’s knowledge base.
